# Maratończycy: analiza w stylu `study + realworld`

Ten notebook rozwija demo `maratonczycy_groupby_pivot_demo.ipynb`, ale w stylu bardziej zbliżonym do:
- `wyklad_analiza_1_study.ipynb`
- `wyklad_analiza_1_realworldexploration.ipynb`

Cel:
1. przygotować dane,
2. obejrzeć rozkłady,
3. zbadać relacje między zmiennymi,
4. zrobić sensowne `groupby` i `pivot_table`,
5. wyciągnąć kilka czytelnych wniosków.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

candidate_paths = [
    Path('marathon-data.csv'),
    Path('/mnt/data/marathon-data.csv'),
    Path('../marathon-data.csv'),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Nie znaleziono pliku marathon-data.csv')

df = pd.read_csv(DATA_PATH)
print(DATA_PATH)
df.head()


## 1. Podstawowy przegląd danych

In [ ]:

df.info()


In [ ]:

df.describe(include='all').transpose()


## 2. Przygotowanie danych
Konwertujemy czasy na `timedelta` i minuty. Dodajemy też `slowdown_min`, czyli ile minut zawodnik stracił względem idealnie równego tempa na drugiej połowie.

In [ ]:

for col in ['split', 'final']:
    df[col + '_td'] = pd.to_timedelta(df[col])
    df[col + '_min'] = df[col + '_td'].dt.total_seconds() / 60

df['slowdown_min'] = df['final_min'] - 2 * df['split_min']
df['slowdown_pct'] = 100 * df['slowdown_min'] / df['final_min']
df['negative_split'] = df['slowdown_min'] < 0

age_bins = [15, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 100]
age_labels = ['16-19', '20-24', '25-29', '30-34', '35-39', '40-44',
              '45-49', '50-54', '55-59', '60-64', '65-69', '70+']

df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)

df.head()


## 3. Podstawowe statystyki liczbowe

In [ ]:

df[['split_min', 'final_min', 'slowdown_min', 'slowdown_pct']].describe().round(2)


## 4. Funkcje pomocnicze do rozkładu i gęstości

In [ ]:

def show_distribution(var_data, title='Rozkład', bins=30):
    fig, ax = plt.subplots(2, 1, figsize=(10, 4))
    ax[0].hist(var_data, bins=bins)
    ax[0].set_title(title)
    ax[0].axvline(var_data.mean(), color='cyan', linestyle='--', linewidth=2, label='mean')
    ax[0].axvline(var_data.median(), color='red', linestyle='--', linewidth=2, label='median')
    ax[0].legend()

    ax[1].boxplot(var_data, vert=False)
    ax[1].set_xlabel('wartość')
    plt.tight_layout()
    plt.show()

def show_density(var_data, title='Gęstość'):
    plt.figure(figsize=(10, 4))
    var_data.plot.density()
    plt.title(title)
    plt.axvline(var_data.mean(), color='cyan', linestyle='--', linewidth=2, label='mean')
    plt.axvline(var_data.median(), color='red', linestyle='--', linewidth=2, label='median')
    plt.legend()
    plt.show()


## 5. Rozkład czasu końcowego

In [ ]:

show_distribution(df['final_min'], title='Czas końcowy [min]')
show_density(df['final_min'], title='Gęstość czasu końcowego [min]')


## 6. Histogram i PDF
Krótka ciekawostka statystyczna: histogram jest dyskretny, a PDF to gładkie przybliżenie rozkładu ciągłego.

In [ ]:

mu = df['final_min'].mean()
sigma = df['final_min'].std(ddof=1)
x = np.linspace(df['final_min'].min() - 10, df['final_min'].max() + 10, 400)
pdf = stats.norm.pdf(x, loc=mu, scale=sigma)

plt.figure(figsize=(10, 4))
plt.hist(df['final_min'], bins=40, density=True, alpha=0.6, label='Histogram')
plt.plot(x, pdf, color='crimson', linewidth=2, label='PDF N(mean, std)')
plt.title('Czas końcowy: histogram i PDF')
plt.xlabel('final_min')
plt.ylabel('gęstość')
plt.legend()
plt.show()

within_one_sigma = ((df['final_min'] >= mu - sigma) & (df['final_min'] <= mu + sigma)).mean()
print(f'Odsetek biegaczy w zakresie mean ± 1 std: {within_one_sigma:.2%}')


## 7. Rozkład spowolnienia (`slowdown_min`)

In [ ]:

show_distribution(df['slowdown_min'], title='Spowolnienie po połowie [min]')
show_density(df['slowdown_min'], title='Gęstość spowolnienia [min]')


## 8. Relacja między split i final
Tutaj bardzo dobrze działa scatter, korelacja i prosta regresji.

In [ ]:

corr = df['split_min'].corr(df['final_min'])
m, b, r, p, se = stats.linregress(df['split_min'], df['final_min'])

x_line = np.linspace(df['split_min'].min(), df['split_min'].max(), 200)
y_line = m * x_line + b

plt.figure(figsize=(7, 5))
plt.scatter(df['split_min'], df['final_min'], s=8, alpha=0.25)
plt.plot(x_line, y_line, color='crimson', linewidth=2)
plt.title(f'Split vs Final | corr={corr:.3f}')
plt.xlabel('split_min')
plt.ylabel('final_min')
plt.show()

print(f'm = {m:.4f}, b = {b:.4f}, r = {r:.4f}, p = {p:.4g}')


## 9. Boxplot czasu końcowego względem płci

In [ ]:

df.boxplot(column='final_min', by='gender', figsize=(8, 5))
plt.title('Czas końcowy według płci')
plt.suptitle('')
plt.xlabel('gender')
plt.ylabel('final_min')
plt.show()


## 10. `groupby`: grupy wiekowe
Liczymy liczebność, średnią, medianę i przeciętne spowolnienie.

In [ ]:

age_summary = (
    df.groupby('age_group', observed=False)
      .agg(
          n=('final_min', 'size'),
          mean_final=('final_min', 'mean'),
          median_final=('final_min', 'median'),
          mean_split=('split_min', 'mean'),
          mean_slowdown=('slowdown_min', 'mean'),
          mean_slowdown_pct=('slowdown_pct', 'mean'),
          negative_split_share=('negative_split', 'mean')
      )
      .reset_index()
)

age_summary.round(3)


## 11. Które grupy wiekowe są najszybsze?

In [ ]:

print('Ranking po średniej:')
display(age_summary.sort_values('mean_final').head(5).round(2))

print('Ranking po medianie:')
display(age_summary.sort_values('median_final').head(5).round(2))


## 12. Uwaga na wiarygodność: liczebność grup
To jest bardzo ważny niuans interpretacyjny.

In [ ]:

plt.figure(figsize=(10, 4))
plt.bar(age_summary['age_group'].astype(str), age_summary['n'], color='slateblue')
plt.title('Liczebność grup wiekowych')
plt.xlabel('age_group')
plt.ylabel('n')
plt.xticks(rotation=35)
plt.show()

age_summary[['age_group', 'n']].round(0)


## 13. `pivot_table`: wiek × płeć

In [ ]:

pivot_mean = pd.pivot_table(
    df,
    index='age_group',
    columns='gender',
    values='final_min',
    aggfunc='mean',
    observed=False
)

pivot_median = pd.pivot_table(
    df,
    index='age_group',
    columns='gender',
    values='final_min',
    aggfunc='median',
    observed=False
)

pivot_count = pd.pivot_table(
    df,
    index='age_group',
    columns='gender',
    values='final_min',
    aggfunc='size',
    observed=False
)

print('Średnia:')
display(pivot_mean.round(2))
print('Mediana:')
display(pivot_median.round(2))
print('Liczebność:')
display(pivot_count)


## 14. `pivot_table` dla spowolnienia

In [ ]:

pivot_slow = pd.pivot_table(
    df,
    index='age_group',
    columns='gender',
    values='slowdown_min',
    aggfunc='mean',
    observed=False
)

pivot_slow.round(2)


## 15. Wykres liniowy: średnia i mediana po grupach

In [ ]:

plt.figure(figsize=(10, 4))
plt.plot(age_summary['age_group'].astype(str), age_summary['mean_final'], marker='o', label='mean_final')
plt.plot(age_summary['age_group'].astype(str), age_summary['median_final'], marker='o', label='median_final')
plt.xticks(rotation=35)
plt.ylabel('minuty')
plt.title('Czas końcowy vs grupa wiekowa')
plt.legend()
plt.show()


## 16. Wykres słupkowy: średni wynik `age_group × gender`

In [ ]:

pivot_mean.round(2).plot(kind='bar', figsize=(10, 4))
plt.ylabel('średni czas końcowy [min]')
plt.title('Średni wynik końcowy: grupa wiekowa × płeć')
plt.show()


## 17. Error bars po grupach wiekowych
Średnia bez rozrzutu bywa myląca, więc dokładamy odchylenie standardowe.

In [ ]:

age_err = (
    df.groupby('age_group', observed=False)
      .agg(mean_final=('final_min', 'mean'), std_final=('final_min', 'std'))
      .reset_index()
)

plt.figure(figsize=(10, 4))
plt.errorbar(
    age_err['age_group'].astype(str),
    age_err['mean_final'],
    yerr=age_err['std_final'],
    fmt='o-',
    capsize=4
)
plt.xticks(rotation=35)
plt.title('Średni czas końcowy ± std')
plt.ylabel('minuty')
plt.show()


## 18. Adnotacja na wykresie
Warto czasem podpisać najciekawszy punkt.

In [ ]:

best_row = age_summary.loc[age_summary['mean_final'].idxmin()]

plt.figure(figsize=(10, 4))
plt.plot(age_summary['age_group'].astype(str), age_summary['mean_final'], marker='o')
plt.annotate(
    f"najszybsza średnia\n{best_row['age_group']}",
    xy=(age_summary['age_group'].astype(str)[best_row.name], best_row['mean_final']),
    xytext=(best_row.name + 0.5, best_row['mean_final'] + 8),
    arrowprops=dict(arrowstyle='->')
)
plt.xticks(rotation=35)
plt.ylabel('minuty')
plt.title('Adnotacja najlepszego wyniku grupowego')
plt.show()


## 19. Krótkie wnioski
Nie chodzi o to, żeby znaleźć jedną „prawdziwą” odpowiedź, tylko żeby pokazać studentom spójny workflow: opis, relacja, grupowanie, pivot, interpretacja.

In [ ]:

summary_points = {
    'najszybsza_grupa_srednia': age_summary.sort_values('mean_final').iloc[0]['age_group'],
    'najszybsza_grupa_mediana': age_summary.sort_values('median_final').iloc[0]['age_group'],
    'najwieksze_spowolnienie_srednio': age_summary.sort_values('mean_slowdown', ascending=False).iloc[0]['age_group'],
    'korelacja_split_final': round(corr, 3),
    'odsetek_negative_split': round(df['negative_split'].mean() * 100, 2)
}
summary_points


## 20. Pytania / ćwiczenia dla studentów

1. Która grupa wiekowa jest najszybsza po średniej, a która po medianie?  
2. Czy ranking zmienia się po rozbiciu na płeć?  
3. Które grupy najbardziej zwalniają po połowie dystansu?  
4. Dlaczego sama średnia może być myląca bez liczebności grup?  
5. Jak zmieni się interpretacja, jeśli zamiast `mean` użyjemy `median`?


In [ ]:
# TODO dla studentów:
# 1. Zrób pivot_table dla negative_split po age_group i gender
# 2. Posortuj age_summary po mean_slowdown_pct
# 3. Narysuj własny wykres porównujący kobiety i mężczyzn w grupach wiekowych